In [ ]:
from NBADeepFm import ShotCycle

# PlayerID = NewType('PlayerID', int)



    
    

In [7]:
import pbpstats
from pbpstats.client import Client

settings = {
    "dir": "./response_data",
    "Boxscore": {"source": "file", "data_provider": "stats_nba"},
    "Possessions": {"source": "file", "data_provider": "stats_nba"},
    "EnhancedPbp": {"source": "file", "data_provider": "stats_nba"},
}


client = Client(settings=settings)
game_id = '0021800001'

game = client.Game(game_id)

possession_count = 0
for possession in game.possessions.items[1:]:
    # print(possession.data)

    possession_count += 1
    for event in possession.events:
        # print(event)
        # print(event.base_stats)
        # if isinstance(event, pbpstats.resources.enhanced_pbp.FreeThrow):
        #     # print(event.data)
        #     data = event.data
        #     next_event = data.get('next_event')
            # print(next_event)
            # if isinstance(next_event, pbpstats.resources.enhanced_pbp.Rebound):
            #     # print(next_event.event_stats)
            #     # print(next_event.data)
            #     # rebounder_id = next_event.event_stats.get('player_id', 0)
            #     print(len(next_event.event_stats))
            #     print("possession", possession_count)
            #     print(next_event.event_stats)
            #     print('*' * 10)
            #     break

        if isinstance(event, pbpstats.resources.enhanced_pbp.FieldGoal):
            print(event)
            print(event.data)
            print(event.distance)
            break


        # if isinstance(event, pbpstats.resources.enhanced_pbp.Turnover):
        #     print(event.player1_id)
        #     print("player1id")
        #     try:
        #         print(event.player3_id)
        #         print("player3id")
        #     except:
        #         print("no player3id")

        #     print("was a steal", event.is_steal)
        #     print('-' * 20)
        #     # print(event.)
        #     break

        # if isinstance(event, pbpstats.resources.enhanced_pbp.FieldGoal):
        #      print(event.rebound)
        #      if event.rebound:
        #          print(event.rebound.data['player1_id'])
        #          break






<StatsFieldGoal GameId: 0021800001, Description: MISS Tatum 25' 3PT Jump Shot, Time: 11:15, EventNum: 10>
{'game_id': '0021800001', 'event_num': 10, 'clock': '11:15', 'period': 1, 'event_action_type': 1, 'event_type': 2, 'player1_id': 1628369, 'team_id': 1610612738, 'video_available': 1, 'description': "MISS Tatum 25' 3PT Jump Shot", 'order': 4, 'player_game_fouls': defaultdict(<class 'int'>, {}), 'possession_changing_override': False, 'non_possession_changing_override': False, 'score': defaultdict(<class 'int'>, {}), 'previous_event': <StatsRebound GameId: 0021800001, Description: CELTICS Rebound, Time: 11:40, EventNum: 8>, 'next_event': <StatsRebound GameId: 0021800001, Description: Saric REBOUND (Off:0 Def:1), Time: 11:13, EventNum: 11>, 'fouls_to_give': defaultdict(<function NbaEnhancedPbpLoader._add_extra_attrs_to_all_events.<locals>.<lambda> at 0x75ad0d49ce00>, {}), 'locX': -148, 'locY': 207}
25.4
<StatsFieldGoal GameId: 0021800001, Description: MISS Brown 2' Running Layup, Time:

In [8]:
from pbpstats.objects.game import Game
from pbpstats.resources.enhanced_pbp import FieldGoal, Turnover, Rebound, FreeThrow, Foul, enhanced_pbp_item
from typing import Tuple, List


def get_current_players(event: enhanced_pbp_item, offensive_team_id, defensive_team_id) -> Tuple[List[PlayerID] | None, List[PlayerID] | None]:
    try:
        lineup: dict = event.current_players
        offensive_players: List[str] = lineup[offensive_team_id]
        defensive_players: List[str] = lineup[defensive_team_id]

        offensive_players: List[PlayerID] = [PlayerID(int(player)) for player in offensive_players]
        defensive_players: List[PlayerID] = [PlayerID(int(player)) for player in defensive_players]
        return offensive_players, defensive_players        
    except KeyError:
        return None, None

# long ahh function
def create_shot_cycles(game: Game) -> List[ShotCycle] | None:
    shot_cycles: List[ShotCycle] = []

    for possession in game.possessions.items:
        events = possession.events



        try:
            team_ids = possession.get_team_ids()
        except Exception as e:
            continue
        
        offensive_team_id = possession.offense_team_id
        defensive_team_id = team_ids[0] if team_ids[1] == offensive_team_id else team_ids[1]
    
        field_goal_attempts = [ev for ev in events if isinstance(ev, FieldGoal)]
        # there should be an event after every missed field goal
        for fg in field_goal_attempts:
            offensive_players, defensive_players = get_current_players(fg, offensive_team_id, defensive_team_id)

            if not offensive_players or not defensive_players:
                continue

            shooting_player = fg.shot_data.get('PlayerId')
            shot_distance = fg.distance
            points_scored = fg.shot_value if fg.is_made else 0
            assisting_player = fg.shot_data.get('AssistPlayerId') if fg.is_assisted else None
            defending_player = fg.shot_data.get('BlockPlayerId') if fg.is_blocked else None
            rebounding_player = None
            is_and1 = fg.is_and1
            is_putback = fg.is_putback

            # current_event_num = fg.data['event_num']
            current_event_num = fg.event_num

            if is_and1:
                first_ft_after_fg = next((ev for ev in events if isinstance(ev, FreeThrow) and ev.data['event_num'] > current_event_num), None)
                if first_ft_after_fg is not None:
                    if first_ft_after_fg.is_made:
                        # allat for adding a point score
                        points_scored += 1
                    else:
                        # get rebounding player
                        # rebound = first_ft_after_fg.rebound
                        ft_event_num = first_ft_after_fg.event_num
                        rebound = next((ev for ev in events if isinstance(ev, Rebound) and int(ev.data['event_num']) > ft_event_num), None)
                        if rebound is not None:
                            rebounding_player = rebound.data['player1_id']
                            rebounding_player = rebounding_player
            elif not fg.is_made:
                try:
                    rebound = fg.rebound
                    rebounding_player = rebound.data['player1_id']
                except Exception as e:
                    continue

                # ? what to do to distinguish from a ft rebound and a regular rebound on and1s

            shot_cycle = ShotCycle(
                offensive_players=offensive_players,
                defensive_players=defensive_players,
                shooting_player=shooting_player,
                shot_distance=shot_distance,
                assisting_player=assisting_player,
                defending_player=defending_player,
                rebounding_player=rebounding_player,
                is_and1=is_and1,
                is_putback=is_putback,
                shot_points_scored=points_scored
            )

            shot_cycles.append(shot_cycle)


        # set of free throws modeled as a single shot cycle
        fts = [ev for ev in events if isinstance(ev, FreeThrow)]

        foul_to_ft_map = {}

         # group fts by foul    
        for ft in fts:
            if ft.is_technical_ft:
                continue
            foul = ft.foul_that_led_to_ft
            if foul is not None:
                foul_to_ft_map.setdefault(foul, []).append(ft)


        for foul, fts_from_foul_list in foul_to_ft_map.items():
            foul: Foul = foul
            is_and1_ft = len(fts_from_foul_list) == 1

            if is_and1_ft:
                # skip it will be checked for in the field goals
                continue
            else:
                # regular fts
                # create a shot cycle based off the free throws

                points_scored = sum([1 if ft.is_made else 0 for ft in fts_from_foul_list])
                # need to get players at the time of the foul
                offensive_players, defensive_players = get_current_players(foul, offensive_team_id, defensive_team_id)

                if not offensive_players or not defensive_players:
                    continue


                shooting_player = PlayerID(foul.player1_id)
                defending_player = None
                try:
                    defending_player = PlayerID(foul.player3_id)
                except Exception as e:
                    pass


                rebounding_player = None

                last_ft: FreeThrow = fts_from_foul_list[-1]

                if last_ft and not last_ft.is_made:
                    # find rebounding player
                    data: dict = last_ft.data
                    next_event = data.get('next_event')
                    # print(next_event)
                    if isinstance(next_event, Rebound):
                        rebounding_player = next_event.player1_id

                shot_cycle = ShotCycle(
                    offensive_players=offensive_players,
                    defensive_players=defensive_players,
                    shooting_player=shooting_player,
                    defending_player=defending_player,
                    is_freethrow=True,
                    rebounding_player=rebounding_player,
                    shot_points_scored=points_scored,
                )

                shot_cycles.append(shot_cycle)

        turnovers = [ev for ev in events if isinstance(ev, Turnover)]
        for to in turnovers:
            offensive_players, defensive_players = get_current_players(to, offensive_team_id, defensive_team_id)

            if not offensive_players or not defensive_players:
                continue

            # get defending player because they caused a steal
            turnover_id = to.player1_id
            try:
                defending_player = to.player3_id
            except Exception as e:
                defending_player = None

            shot_cycle = ShotCycle(
                offensive_players=offensive_players,
                defensive_players=defensive_players,
                shooting_player=turnover_id,
                defending_player=defending_player,
                is_steal=to.is_steal,
                is_turnover=True,
                shot_points_scored=0
            )
            shot_cycles.append(shot_cycle)


    return shot_cycles



    

In [17]:
import polars as pl
from tqdm import tqdm
import traceback

def construct_df_for_season(game_ids: list[str]) -> pl.DataFrame:
    shot_cycles_for_season = []
    for game_id in tqdm(game_ids):
        try:
            game_id = game_id.zfill(10)
            game = client.Game(game_id)
            shot_cycles_for_game = create_shot_cycles(game)
            shot_cycles_for_season.extend(shot_cycles_for_game)

            # print(f'added {len(shot_cycles_for_game)} for {game_id}')

        except pbpstats.data_loader.stats_nba.possessions.loader.TeamHasBackToBackPossessionsException:
            # print(f'no enhanced pbp for {game_id}')
            continue
        
        except Exception as e:
            print(e)
            print(traceback.format_exc())
            continue

    print('Have', len(shot_cycles_for_season), 'shot cycles for season')
    season_df = pl.DataFrame(data=shot_cycles_for_season)



    return season_df






In [10]:
# fetch all the seasons
season = '22018'

games_df = pl.read_csv('csv/game.csv')
games_df.head()





season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,fga_home,fg_pct_home,fg3m_home,fg3a_home,fg3_pct_home,ftm_home,fta_home,ft_pct_home,oreb_home,dreb_home,reb_home,ast_home,stl_home,blk_home,tov_home,pf_home,pts_home,plus_minus_home,video_available_home,team_id_away,team_abbreviation_away,team_name_away,matchup_away,wl_away,fgm_away,fga_away,fg_pct_away,fg3m_away,fg3a_away,fg3_pct_away,ftm_away,fta_away,ft_pct_away,oreb_away,dreb_away,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
i64,i64,str,str,i64,str,str,str,i64,f64,f64,f64,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,i64,i64,i64,str,str,str,str,f64,f64,f64,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,i64,i64,str
21946,1610610035,"""HUS""","""Toronto Huskies""",24600001,"""1946-11-01 00:00:00""","""HUS vs. NYK""","""L""",0,25.0,null,null,null,null,null,16.0,29.0,0.552,null,null,null,null,null,null,null,null,66.0,-2,0,1610612752,"""NYK""","""New York Knicks""","""NYK @ HUS""","""W""",24.0,null,null,null,null,null,20.0,26.0,0.769,null,null,null,null,null,null,null,null,68.0,2,0,"""Regular Season"""
21946,1610610034,"""BOM""","""St. Louis Bombers""",24600003,"""1946-11-02 00:00:00""","""BOM vs. PIT""","""W""",0,20.0,59.0,0.339,null,null,null,16.0,null,null,null,null,null,null,null,null,null,21.0,56.0,5,0,1610610031,"""PIT""","""Pittsburgh Ironmen""","""PIT @ BOM""","""L""",16.0,72.0,0.222,null,null,null,19.0,null,null,null,null,null,null,null,null,null,25.0,51.0,-5,0,"""Regular Season"""
21946,1610610032,"""PRO""","""Providence Steamrollers""",24600002,"""1946-11-02 00:00:00""","""PRO vs. BOS""","""W""",0,21.0,null,null,null,null,null,17.0,null,null,null,null,null,null,null,null,null,null,59.0,6,0,1610612738,"""BOS""","""Boston Celtics""","""BOS @ PRO""","""L""",21.0,null,null,null,null,null,11.0,null,null,null,null,null,null,null,null,null,null,53.0,-6,0,"""Regular Season"""
21946,1610610025,"""CHS""","""Chicago Stags""",24600004,"""1946-11-02 00:00:00""","""CHS vs. NYK""","""W""",0,21.0,null,null,null,null,null,21.0,null,null,null,null,null,null,null,null,null,20.0,63.0,16,0,1610612752,"""NYK""","""New York Knicks""","""NYK @ CHS""","""L""",16.0,null,null,null,null,null,15.0,null,null,null,null,null,null,null,null,null,22.0,47.0,-16,0,"""Regular Season"""
21946,1610610028,"""DEF""","""Detroit Falcons""",24600005,"""1946-11-02 00:00:00""","""DEF vs. WAS""","""L""",0,10.0,null,null,null,null,null,13.0,null,null,null,null,null,null,null,null,null,null,33.0,-17,0,1610610036,"""WAS""","""Washington Capitols""","""WAS @ DEF""","""W""",18.0,null,null,null,null,null,14.0,null,null,null,null,null,null,null,null,null,null,50.0,17,0,"""Regular Season"""


In [11]:
# filter for only regular season games post 2000
games_df = games_df.filter(pl.col('season_id').cast(pl.String).str.starts_with('22'))
season_ids = games_df.select(pl.col('season_id').unique())
season_ids = season_ids['season_id'].to_list()
season_ids = list(reversed(season_ids))
print(season_ids)

[22022, 22021, 22020, 22019, 22018, 22017, 22016, 22015, 22014, 22013, 22011, 22010, 22009, 22008, 22007, 22006, 22005, 22004, 22003, 22002, 22001, 22000]


In [18]:
for season_id in season_ids[1:]:
    print(season_id)
    season_games_df = games_df.filter(pl.col('season_id') == season_id)
    game_ids_for_season = season_games_df['game_id'].cast(pl.String).unique().to_list()
    season_df = construct_df_for_season(game_ids_for_season)
    # shot_cycles_df.write_csv(f'gen_data/{season_id}_shot_cycles.csv')
    season_df.write_avro(f'gen_data/{season_id}_shot_cycles.avro')





22021


100%|██████████| 1230/1230 [01:00<00:00, 20.25it/s]


Have 269441 shot cycles for season
22020


100%|██████████| 1080/1080 [00:57<00:00, 18.78it/s]


Have 237908 shot cycles for season
22019


...

Have 250329 shot cycles for season
22001


100%|██████████| 1189/1189 [01:04<00:00, 18.51it/s]


Have 249614 shot cycles for season
22000


100%|██████████| 1189/1189 [01:07<00:00, 17.72it/s]


Have 250870 shot cycles for season
